# results_calib
Quick summary notebook for calibration results.
Prints mean values and permutation p-values in a readable format.


In [1]:
import pandas as pd
import numpy as np
from scipy.stats import permutation_test
import sys
import os


In [13]:
def paired_permutation_tests(df, value_col, model_a, model_b):
    results = []

    for task, group in df.groupby("task"):
        g = group[group["model"].isin([model_a, model_b])]

        pivoted = g.pivot(index="subject", columns="model", values=value_col).dropna()

        if pivoted.shape[0] < 2:
            continue

        a = pivoted[model_a].values
        b = pivoted[model_b].values

        res = permutation_test(
            (a, b),
            statistic=lambda x, y: (x - y).mean(),
            permutation_type="samples",
            alternative="two-sided",
            n_resamples=10000,
        )

        results.append({
            "task": task,
            "model_a": model_a,
            "model_b": model_b,
            "mean_diff": (a - b).mean(),
            "p_value": res.pvalue,
            "n": len(pivoted)
        })

    return pd.DataFrame(results)


In [3]:
subjects = ["sub-02","sub-03","sub-04", "sub-05", "sub-06", "sub-07","sub-08", "sub-09","sub-10", "sub-11", "sub-13", "sub-14", "sub-15", "sub-16", "sub-17"]

In [4]:
# Personal imports
deepmreye_root = os.path.abspath(
    os.path.join(os.getcwd(), "..", "..")
)

main_dir = "/Users/sinakling/disks/meso_shared"
project_dir = "deepmreye"

sys.path.append("{}/analysis_code/utils".format(deepmreye_root))
from stats_utils import *
sys.path.append("{}/training_code/utils".format(deepmreye_root))
from eyetrack_utils import chunk_and_median, adapt_evaluation

In [5]:
# Pretrained DeepMReye
df_ee_pretrained = compute_ee_summary(subjects, tasks=('fixation', 'pursuit', 'freeview', 'all'), runs=('01', '02', '03'), model_name='pretrained',base_path="/Users/sinakling/disks/meso_shared/deepmreye/derivatives/pp_data",ci95=False)
print(df_ee_pretrained.head())

# Scaled DeepMReye 
df_ee_scaled = compute_ee_summary(subjects, tasks=('fixation', 'pursuit', 'freeview', 'all'), runs=('01', '02', '03'), model_name='scaled',base_path="/Users/sinakling/disks/meso_shared/deepmreye/derivatives/pp_data",ci95=False)

# DeepMReye Calibrated no_interpol
df_ee_finetuned = compute_ee_summary(subjects, tasks=('fixation', 'pursuit', 'freeview', 'all'), runs=('01', '02', '03'), model_name='no_interpol',base_path="/Users/sinakling/disks/meso_shared/deepmreye/derivatives/pp_data",ci95=False)

# 10 dva #p
df_ee_pt_5deg = compute_ee_summary(subjects, tasks=('fixation', 'pursuit', 'freeview', 'all'), runs=('01', '02', '03'), model_name='pt_fivedegree',base_path="/Users/sinakling/disks/meso_shared/deepmreye/derivatives/pp_data",ci95=False)
df_ee_ft_5deg = compute_ee_summary(subjects, tasks=('fixation', 'pursuit', 'freeview', 'all'), runs=('01', '02', '03'), model_name='ft_fivedegree',base_path="/Users/sinakling/disks/meso_shared/deepmreye/derivatives/pp_data",ci95=False)

# Eye tracking vs target coordinates ft_sim
df_ee_sim = compute_ee_summary(subjects, tasks=('fixation', 'pursuit', 'freeview', 'all'), runs=('01', '02', '03'), model_name='ft_sim',base_path="/Users/sinakling/disks/meso_shared/deepmreye/derivatives/pp_data",ci95=False)
print(df_ee_sim.head())

  subject      task       model      mean   75_perc  n_samples
0  sub-02  fixation  pretrained  5.256241  6.885061        150
1  sub-02   pursuit  pretrained  4.288809  5.481960        162
2  sub-02  freeview  pretrained  4.147764  5.291535         90
3  sub-02       all  pretrained  4.436713  5.585948        462
4  sub-03  fixation  pretrained  5.427978  7.019215        150
  subject      task   model      mean   75_perc  n_samples
0  sub-02  fixation  ft_sim  3.910202  5.189747        150
1  sub-02   pursuit  ft_sim  4.213809  5.286843        162
2  sub-02  freeview  ft_sim  3.818474  5.017093         90
3  sub-02       all  ft_sim  3.897994  5.032605        462
4  sub-03  fixation  ft_sim  3.739895  4.679313        150


In [11]:
df_ee = pd.concat([df_ee_pretrained, df_ee_scaled, df_ee_finetuned, df_ee_pt_5deg, df_ee_ft_5deg, df_ee_sim], ignore_index=True)

## EE summary

In [7]:
def bootstrap_ci_mean(data, n_bootstrap=1000, ci_level=0.95):
    n = len(data)
    bootstrap_samples = np.random.choice(data, size=(n_bootstrap, n), replace=True)
    means = np.mean(bootstrap_samples, axis=1)
    lower_ci = np.percentile(means, (1 - ci_level) / 2 * 100)
    upper_ci = np.percentile(means, (1 + ci_level) / 2 * 100)
    return lower_ci, upper_ci

ee_summary = (
    df_ee
    .groupby(["model", "task"])["mean"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

# Compute bootstrap CI per group
ci_values = df_ee.groupby(["model", "task"])["mean"].apply(
    lambda x: bootstrap_ci_mean(x.values)
).reset_index()
ci_values[["ci95_low", "ci95_high"]] = pd.DataFrame(ci_values["mean"].tolist(), index=ci_values.index)
ci_values = ci_values.drop(columns="mean")

ee_summary = ee_summary.merge(ci_values, on=["model", "task"])

print("===== EE VALUES =====")
for _, r in ee_summary.iterrows():
    print(
        f"Task={r.task:12s} | Model={r.model:15s} | "
        f"Mean={r['mean']:.2f} | SD={r['std']:.2f} | "
        f"CI=[{r['ci95_low']:.2f}, {r['ci95_high']:.2f}] | "
        f"N={int(r['count'])}"
    )



===== EE VALUES =====
Task=all          | Model=ft_fivedegree   | Mean=2.55 | SD=0.50 | CI=[2.32, 2.81] | N=15
Task=fixation     | Model=ft_fivedegree   | Mean=3.42 | SD=1.09 | CI=[2.94, 4.00] | N=15
Task=freeview     | Model=ft_fivedegree   | Mean=3.26 | SD=0.65 | CI=[2.96, 3.56] | N=15
Task=pursuit      | Model=ft_fivedegree   | Mean=2.11 | SD=0.48 | CI=[1.91, 2.38] | N=15
Task=all          | Model=ft_sim          | Mean=3.14 | SD=0.65 | CI=[2.80, 3.44] | N=15
Task=fixation     | Model=ft_sim          | Mean=3.89 | SD=1.11 | CI=[3.37, 4.43] | N=15
Task=freeview     | Model=ft_sim          | Mean=4.09 | SD=1.07 | CI=[3.57, 4.61] | N=15
Task=pursuit      | Model=ft_sim          | Mean=2.48 | SD=0.70 | CI=[2.18, 2.86] | N=15
Task=all          | Model=no_interpol     | Mean=3.12 | SD=0.64 | CI=[2.82, 3.43] | N=15
Task=fixation     | Model=no_interpol     | Mean=4.04 | SD=1.11 | CI=[3.47, 4.60] | N=15
Task=freeview     | Model=no_interpol     | Mean=3.63 | SD=0.83 | CI=[3.23, 4.02] | N=15

## Correlation summary

In [8]:
# load correlation dataframes

df_corr_base = pd.read_csv(f'{main_dir}/{project_dir}/derivatives/pp_data/group/correlations/correlation_calib.csv')
df_corr_5deg = pd.read_csv(f'{main_dir}/{project_dir}/derivatives/pp_data/group/correlations/correlation_5deg_calib.csv')
df_corr_sim = pd.read_csv(f'{main_dir}/{project_dir}/derivatives/pp_data/group/correlations/correlation_sim_calib.csv')

df_corr = pd.concat([df_corr_base, df_corr_5deg, df_corr_sim], ignore_index=True)

In [9]:
corr_summary = (
    df_corr
    .groupby(["model", "task"])["mean_pearson"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

# Compute bootstrap CI per group
ci_values = df_corr.groupby(["model", "task"])["mean_pearson"].apply(
    lambda x: bootstrap_ci_mean(x.values)
).reset_index()
ci_values[["ci95_low", "ci95_high"]] = pd.DataFrame(ci_values["mean_pearson"].tolist(), index=ci_values.index)
ci_values = ci_values.drop(columns="mean_pearson")

corr_summary = corr_summary.merge(ci_values, on=["model", "task"])

print("===== CORRELATION MEANS =====")
for _, r in corr_summary.iterrows():
    print(
        f"Task={r.task:12s} | Model={r.model:15s} | "
        f"Mean={r['mean']:.2f} | SD={r['std']:.2f} | "
        f"CI=[{r['ci95_low']:.2f}, {r['ci95_high']:.2f}] | "
        f"N={int(r['count'])}"
    )

===== CORRELATION MEANS =====
Task=all          | Model=ft_fivedegree   | Mean=0.72 | SD=0.08 | CI=[0.68, 0.76] | N=15
Task=fixation     | Model=ft_fivedegree   | Mean=0.75 | SD=0.14 | CI=[0.68, 0.81] | N=15
Task=freeview     | Model=ft_fivedegree   | Mean=0.61 | SD=0.13 | CI=[0.54, 0.67] | N=15
Task=pursuit      | Model=ft_fivedegree   | Mean=0.88 | SD=0.04 | CI=[0.86, 0.89] | N=15
Task=all          | Model=no_interpol     | Mean=0.85 | SD=0.05 | CI=[0.83, 0.88] | N=15
Task=fixation     | Model=no_interpol     | Mean=0.87 | SD=0.07 | CI=[0.84, 0.91] | N=15
Task=freeview     | Model=no_interpol     | Mean=0.73 | SD=0.14 | CI=[0.66, 0.79] | N=15
Task=pursuit      | Model=no_interpol     | Mean=0.93 | SD=0.03 | CI=[0.91, 0.94] | N=15
Task=all          | Model=pretrained      | Mean=0.81 | SD=0.07 | CI=[0.78, 0.85] | N=15
Task=fixation     | Model=pretrained      | Mean=0.84 | SD=0.09 | CI=[0.79, 0.88] | N=15
Task=freeview     | Model=pretrained      | Mean=0.70 | SD=0.13 | CI=[0.62, 0.75

## EE permutation tests

In [16]:
pairs = [
    ("pretrained", "scaled"),
    ("pretrained", "no_interpol"),
    ("scaled", "no_interpol"),
    ("pt_fivedegree", "ft_fivedegree")
]

for a, b in pairs:
    res = paired_permutation_tests(df_ee, "mean", a, b)
    if len(res) == 0:
        continue

    print(f"\n--- EE: {a} vs {b} ---")
    for _, r in res.iterrows():
        print(f"Task={r.task:12s} | Δ={r.mean_diff:.4f} | p={r.p_value:.6f} | n={int(r.n)}")



--- EE: pretrained vs scaled ---
Task=all          | Δ=0.2364 | p=0.001000 | n=15
Task=fixation     | Δ=0.6035 | p=0.000400 | n=15
Task=freeview     | Δ=-0.1620 | p=0.057194 | n=15
Task=pursuit      | Δ=0.3063 | p=0.000400 | n=15

--- EE: pretrained vs no_interpol ---
Task=all          | Δ=0.4977 | p=0.000200 | n=15
Task=fixation     | Δ=1.1351 | p=0.000400 | n=15
Task=freeview     | Δ=0.1290 | p=0.169783 | n=15
Task=pursuit      | Δ=0.4039 | p=0.000200 | n=15

--- EE: scaled vs no_interpol ---
Task=all          | Δ=0.2613 | p=0.000400 | n=15
Task=fixation     | Δ=0.5316 | p=0.000800 | n=15
Task=freeview     | Δ=0.2910 | p=0.006199 | n=15
Task=pursuit      | Δ=0.0975 | p=0.082392 | n=15

--- EE: pt_fivedegree vs ft_fivedegree ---
Task=all          | Δ=-0.2196 | p=0.000800 | n=15
Task=fixation     | Δ=-0.3911 | p=0.008999 | n=15
Task=freeview     | Δ=-0.1668 | p=0.088791 | n=15
Task=pursuit      | Δ=-0.1243 | p=0.037196 | n=15


## Correlation permutation tests

In [15]:
pairs = [
    ("pretrained", "scaled"),
    ("pretrained", "no_interpol"),
    ("scaled", "no_interpol"),
    ("pt_fivedegree", "ft_fivedegree")
]

for a, b in pairs:
    res = paired_permutation_tests(df_corr, "mean_pearson", a, b)
    if len(res) == 0:
        continue

    print(f"\n--- CORR: {a} vs {b} ---")
    for _, r in res.iterrows():
        print(f"Task={r.task:12s} | Δ={r.mean_diff:.4f} | p={r.p_value:.6f} | n={int(r.n)}")



--- CORR: pretrained vs scaled ---
Task=all          | Δ=0.0000 | p=0.625937 | n=15
Task=fixation     | Δ=0.0000 | p=0.590341 | n=15
Task=freeview     | Δ=0.0000 | p=0.210179 | n=15
Task=pursuit      | Δ=0.0000 | p=0.916908 | n=15

--- CORR: pretrained vs no_interpol ---
Task=all          | Δ=-0.0376 | p=0.000600 | n=15
Task=fixation     | Δ=-0.0381 | p=0.000600 | n=15
Task=freeview     | Δ=-0.0309 | p=0.004200 | n=15
Task=pursuit      | Δ=-0.0117 | p=0.020798 | n=15

--- CORR: scaled vs no_interpol ---
Task=all          | Δ=-0.0376 | p=0.000400 | n=15
Task=fixation     | Δ=-0.0381 | p=0.000400 | n=15
Task=freeview     | Δ=-0.0309 | p=0.005000 | n=15
Task=pursuit      | Δ=-0.0117 | p=0.025797 | n=15

--- CORR: pt_fivedegree vs ft_fivedegree ---
Task=all          | Δ=-0.0358 | p=0.006799 | n=15
Task=fixation     | Δ=-0.0621 | p=0.004800 | n=15
Task=freeview     | Δ=-0.0233 | p=0.066193 | n=15
Task=pursuit      | Δ=-0.0209 | p=0.118388 | n=15


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Define color and name mappings
model_name_map = {
    "pretrained": "DeepMreye",
    "scaled": "DeepMreye Scaled",
    "no_interpol": "DeepMreye + Calib"
}

model_keys = ["pretrained", "no_interpol", "scaled"]
model_labels = [model_name_map[m] for m in model_keys]

bar_colors = [
    'rgba(165, 209, 216, 0.4)',
    'rgba(24, 151, 178, 0.4)',
    'rgba(48, 109, 116, 0.4)'
]

task_labels = ["<b>Guided Fixation</b>", "<b>Smooth Pursuit</b>", "<b>Freeviewing</b>"]
task_names = ["fixation", "pursuit", "freeview"]

x_positions = [0, 1, 2]  # one per model
bar_width = 0.35

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=task_labels
)

shown_subjects = set()

for j, task in enumerate(task_names):
    col = j + 1

    df_plot = df_corr[
        (df_corr["task"] == task) &
        (df_corr["model"].isin(model_keys))
    ]

    df_pivot = df_plot.pivot(index="subject", columns="model", values="mean_pearson")[model_keys].dropna()

    # Add bars per model
    for k, (model_key, bar_color) in enumerate(zip(model_keys, bar_colors)):
        mean_val = df_pivot[model_key].mean()
        stderr_val = df_pivot[model_key].sem()

        fig.add_trace(go.Bar(
            x=[x_positions[k]],
            y=[mean_val],
            error_y=dict(type="data", array=[stderr_val]),
            marker_color=bar_color,
            marker_line=dict(color=bar_color.replace('0.4', '0.9'), width=1.5),
            width=bar_width,
            name=model_labels[k],
            showlegend=(j == 0),  # only show legend once
            legendgroup=model_key,
        ), row=1, col=col)

    # Add subject lines in grey with transparency
    for subject, row_data in df_pivot.iterrows():
        show_legend = subject not in shown_subjects
        shown_subjects.add(subject)

        fig.add_trace(go.Scatter(
            x=x_positions,
            y=[row_data[m] for m in model_keys],
            mode='lines+markers',
            line=dict(color='rgba(100, 100, 100, 0.35)', width=1.5),
            marker=dict(size=8, color='rgba(100, 100, 100, 0.5)'),
            name=subject,
            showlegend=False,  # remove subject legend, or set True if you want it
            legendgroup=subject,
        ), row=1, col=col)

# Layout
fig.update_layout(
    height=550,
    width=1300,
    title_text="Model Comparison per Task",
    template="simple_white",
    showlegend=True,
    font=dict(size=16, family="Arial"),
    margin=dict(t=120),
    legend=dict(
        title="Model",
        orientation="v",
        x=1.02,
        y=0.5
    )
)

# Axis formatting
for col in range(1, 4):
    fig.update_yaxes(range=[0, 1.1], title_text="Pearson Correlation" if col == 1 else "", row=1, col=col)
    fig.update_xaxes(
        tickvals=x_positions,
        ticktext=model_labels,
        row=1, col=col
    )

fig.show()
fig.write_image("/Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_calibration/figures/group/corelation_plots_individualpoints.pdf")

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# Define color and name mappings
model_name_map = {
    "pretrained": "DeepMreye",
    "scaled": "DeepMreye Scaled",
    "no_interpol": "DeepMreye + Calib"
}

model_keys = ["pretrained", "no_interpol","scaled"]
model_labels = [model_name_map[m] for m in model_keys]

bar_colors = [
    'rgba(165, 209, 216, 0.4)',
    'rgba(24, 151, 178, 0.4)',
    'rgba(48, 109, 116, 0.4)'
]

task_labels = ["<b>Guided Fixation</b>", "<b>Smooth Pursuit</b>", "<b>Freeviewing</b>"]
task_names = ["fixation", "pursuit", "freeview"]

x_positions = [0, 1, 2]  # one per model
bar_width = 0.35

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=task_labels
)

shown_subjects = set()

for j, task in enumerate(task_names):
    col = j + 1

    df_plot = df_ee[
        (df_ee["task"] == task) &
        (df_ee["model"].isin(model_keys))
    ]

    df_pivot = df_plot.pivot(index="subject", columns="model", values="mean")[model_keys].dropna()

    # Add bars per model
    for k, (model_key, bar_color) in enumerate(zip(model_keys, bar_colors)):
        mean_val = df_pivot[model_key].mean()
        stderr_val = df_pivot[model_key].sem()

        fig.add_trace(go.Bar(
            x=[x_positions[k]],
            y=[mean_val],
            error_y=dict(type="data", array=[stderr_val]),
            marker_color=bar_color,
            marker_line=dict(color=bar_color.replace('0.4', '0.9'), width=1.5),
            width=bar_width,
            name=model_labels[k],
            showlegend=(j == 0),  # only show legend once
            legendgroup=model_key,
        ), row=1, col=col)

    # Add subject lines in grey with transparency
    for subject, row_data in df_pivot.iterrows():
        show_legend = subject not in shown_subjects
        shown_subjects.add(subject)

        fig.add_trace(go.Scatter(
            x=x_positions,
            y=[row_data[m] for m in model_keys],
            mode='lines+markers',
            line=dict(color='rgba(100, 100, 100, 0.35)', width=1.5),
            marker=dict(size=8, color='rgba(100, 100, 100, 0.5)'),
            name=subject,
            showlegend=False,  # remove subject legend, or set True if you want it
            legendgroup=subject,
        ), row=1, col=col)

# Layout
fig.update_layout(
    height=550,
    width=1300,
    title_text="Model Comparison per Task",
    template="simple_white",
    showlegend=True,
    font=dict(size=16, family="Arial"),
    margin=dict(t=120),
    legend=dict(
        title="Model",
        orientation="v",
        x=6.02,
        y=0.5
    )
)

# Axis formatting
for col in range(1, 4):
    fig.update_yaxes(range=[0, 7.1], title_text="Euclidean Error (dva)" if col == 1 else "", row=1, col=col)
    fig.update_xaxes(
        tickvals=x_positions,
        ticktext=model_labels,
        row=1, col=col
    )

fig.show()
fig.write_image("/Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_calibration/figures/group/ee_plots_individualpoints.pdf")

In [ ]:
# Individual Example Plots (Figure 2)
import plotly.graph_objects as go
from plotly.subplots import make_subplots

subject = "sub-03"

# Load eye-tracking data and downsample
eye_data = pd.read_csv(f"/Users/sinakling/disks/meso_shared/deepmreye/derivatives/pp_data/{subject}/eyetracking/timeseries/{subject}_task-DeepMReyeCalib_run_01_eyedata.tsv.gz", compression='gzip', delimiter='\t')
eye_data = eye_data[['timestamps', 'x', 'y']].to_numpy()
eye_data_downsampled_x = chunk_and_median(eye_data[:,1])
eye_data_downsampled_y = chunk_and_median(eye_data[:,2])
eye_data_downsampled = np.stack((eye_data_downsampled_x, eye_data_downsampled_y), axis=1)

# Load pretictions 

evaluation_fine_tuned = np.load("/Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_calibration/pred/evaluation_dict_calib_no_interpol.npy", allow_pickle=True).item()
evaluation_pretrained = np.load("/Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_calibration/pred/evaluation_dict_calib_pretrained.npy", allow_pickle=True).item()

subject_data_fine_tuned = evaluation_fine_tuned[f"/scratch/mszinte/data/deepmreye/derivatives/int_deepmreye/deepmreye_calib/pp_data_no_interpol/{subject}_DeepMReyeCalib_label_no_interpol.npz"]
df_pred_median_fine_tuned, df_pred_subtr_fine_tuned = adapt_evaluation(subject_data_fine_tuned)

subject_data_pretrained = evaluation_pretrained[f"/scratch/mszinte/data/deepmreye/derivatives/int_deepmreye/deepmreye_calib/pp_data_pretrained/{subject}_DeepMReyeCalib_no_label.npz"]
df_pred_median_pretrained, df_pred_subtr_pretrained = adapt_evaluation(subject_data_pretrained)




In [ ]:
# Convert TRs to seconds and select Guided Fixation task (+- 2 TRs for plot pretty-ness)
TR = 1.2           # seconds per TR
start_tr = 3       # skip first 5 ITI TRs (-2 TR)
n_trs = 52         # 50 task TRs = 60 seconds (+ 2 TR)

time_axis = np.arange(n_trs) * TR  

plot_rows = 2
plot_cols = 1
fig = make_subplots(rows=plot_rows, cols=plot_cols, shared_xaxes=True, vertical_spacing=0.05)

fig.add_trace(go.Scatter(
    x=time_axis, y=eye_data_downsampled[:,0][start_tr:start_tr + n_trs],
    showlegend=True, name='Eyetracking',
    line=dict(color='#0E1C36', width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=time_axis, y=df_pred_median_pretrained['X'][start_tr:start_tr + n_trs],
    showlegend=True, name='DeepMReye',
    line=dict(color="#A5D1D8", width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=time_axis, y=-1*eye_data_downsampled[:,1][start_tr:start_tr + n_trs],
    showlegend=True, name='Eyetracking',
    line=dict(color='#0E1C36', width=2)
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=time_axis, y=df_pred_median_pretrained['Y'][start_tr:start_tr + n_trs],
    showlegend=True, name='DeepMReye',
    line=dict(color='#A5D1D8', width=2)
), row=2, col=1)


fig.update_traces(showlegend=False, row=1, col=1)

# Format and show fig
fig.update_layout(
    height=600, width=1500,
    template="simple_white",
    title_text=f"Eyetracker Gaze Position (X,Y) vs. Predicted Gaze Position",
    yaxis1=dict(title="<b>Hor. coord. (dva)<b>", title_font=dict(size=15)),
    yaxis2=dict(title="<b>Ver. coord. (dva)<b>", title_font=dict(size=15)),
    xaxis2=dict(title="<b>Time (s)<b>", title_font=dict(size=15))
)

fig.update_yaxes(range=[-15, 15], row=1, col=1)
fig.update_yaxes(range=[-15, 15], row=2, col=1)

fig.update_annotations(font=dict(size=14))
fig.show()
fig.write_image(f"/Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_calibration/figures/{subject}/{subject}_guided_fixation_pretrained.pdf")

In [ ]:
# Individual Example Plots (Figure 2)

# Convert TRs to seconds and select Guided Fixation task (+- 2 TRs for plot pretty-ness)
TR = 1.2           # seconds per TR
start_tr = 3       # skip first 5 ITI TRs (-2 TR)
n_trs = 52         # 50 task TRs = 60 seconds (+ 2 TR)

time_axis = np.arange(n_trs) * TR  

plot_rows = 2
plot_cols = 1
fig = make_subplots(rows=plot_rows, cols=plot_cols, shared_xaxes=True, vertical_spacing=0.05)

fig.add_trace(go.Scatter(
    x=time_axis, y=eye_data_downsampled[:,0][start_tr:start_tr + n_trs],
    showlegend=True, name='Eyetracking',
    line=dict(color='#0E1C36', width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=time_axis, y=df_pred_median_fine_tuned['X'][start_tr:start_tr + n_trs],
    showlegend=True, name='DeepMReye Fine-Tuned',
    line=dict(color="#306D74", width=2)
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=time_axis, y=eye_data_downsampled[:,1][start_tr:start_tr + n_trs],
    showlegend=True, name='Eyetracking',
    line=dict(color='#0E1C36', width=2)
), row=2, col=1)

fig.add_trace(go.Scatter(
    x=time_axis, y=df_pred_median_fine_tuned['Y'][start_tr:start_tr + n_trs],
    showlegend=True, name='DeepMReye Fine-Tuned',
    line=dict(color='#306D74', width=2)
), row=2, col=1)


fig.update_traces(showlegend=False, row=1, col=1)

# Format and show fig
fig.update_layout(
    height=600, width=1500,
    template="simple_white",
    title_text=f"Eyetracker Gaze Position (X,Y) vs. Predicted Gaze Position",
    yaxis1=dict(title="<b>Hor. coord. (dva)<b>", title_font=dict(size=15)),
    yaxis2=dict(title="<b>Ver. coord. (dva)<b>", title_font=dict(size=15)),
    xaxis2=dict(title="<b>Time (s)<b>", title_font=dict(size=15))
)

fig.update_yaxes(range=[-15, 15], row=1, col=1)
fig.update_yaxes(range=[-15, 15], row=2, col=1)

fig.update_annotations(font=dict(size=14))
fig.show()
fig.write_image(f"/Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_calibration/figures/{subject}/{subject}_guided_fixation_fine_tuned.pdf")

In [ ]:
closed_evaluation_pretrained = np.load("/Users/sinakling/disks/meso_shared/deepmreye/derivatives/int_deepmreye/deepmreye_eyestate_tracking/pred/evaluation_dict_pt_gaze_closed.npy", allow_pickle=True).item()
print(closed_evaluation_pretrained.keys())

/scratch/mszinte/derivatives/int_deepmreye/deepmreye_closed/pp_data_pretrained/sub-01_DeepMReyeClosed_no_label.npz

dict_keys(['/scratch/mszinte/data/deepmreye/derivatives/int_deepmreye/deepmreye_closed/pp_data_gaze/sub-14_DeepMReyeClosed_label.npz', '/scratch/mszinte/data/deepmreye/derivatives/int_deepmreye/deepmreye_closed/pp_data_gaze/sub-06_DeepMReyeClosed_label.npz', '/scratch/mszinte/data/deepmreye/derivatives/int_deepmreye/deepmreye_closed/pp_data_gaze/sub-03_DeepMReyeClosed_label.npz', '/scratch/mszinte/data/deepmreye/derivatives/int_deepmreye/deepmreye_closed/pp_data_gaze/sub-07_DeepMReyeClosed_label.npz', '/scratch/mszinte/data/deepmreye/derivatives/int_deepmreye/deepmreye_closed/pp_data_gaze/sub-01_DeepMReyeClosed_label.npz', '/scratch/mszinte/data/deepmreye/derivatives/int_deepmreye/deepmreye_closed/pp_data_gaze/sub-02_DeepMReyeClosed_label.npz', '/scratch/mszinte/data/deepmreye/derivatives/int_deepmreye/deepmreye_closed/pp_data_gaze/sub-08_DeepMReyeClosed_label.npz', '/scratch/mszinte/data/deepmreye/derivatives/int_deepmreye/deepmreye_closed/pp_data_gaze/sub-11_DeepMReyeClosed_label.npz